# P6 — Validación externa de campo

## Objetivo
Evaluar la capacidad de generalización del modelo sobre fotografías propias que no pertenecen al dominio de entrenamiento.

In [1]:

from pathlib import Path
import sys
import yaml

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró config.yaml. "
        "Ejecute Jupyter desde la carpeta del proyecto o notebooks/."
    )

ROOT = find_project_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

cfg = yaml.safe_load(
    (ROOT / "config.yaml").read_text(encoding="utf-8")
)

print("Proyecto:", ROOT)


Proyecto: C:\Users\Nicolas Manco\Downloads\CONCRETE_DL


## 1. Cargar modelo

In [2]:
import torch
from src.models import build_resnet50

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = build_resnet50(
    cfg["model"]["head_hidden"],
    cfg["model"]["dropout"],
    pretrained=False
).to(device)

model.load_state_dict(
    torch.load(
        ROOT/"models/resnet50_finetuned.pt",
        map_location=device
    )
)

model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

## 2. Parámetros espaciales

In [3]:
field_cfg = cfg["field_inference"]

display(field_cfg)

{'patch_size': 224,
 'overlap_fraction': 0.25,
 'small_image_single_patch_ratio': 1.15,
 'aggregation': 'topk_mean',
 'topk_fraction': 0.1,
 'heatmap_alpha': 0.4}

## 3. Inferencia sobre campo y test

In [ ]:
import pandas as pd
from src.inference import predict_field_image, probability_map, aggregate_probabilities
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import hashlib
model_sha256=hashlib.sha256((ROOT/"models/resnet50_finetuned.pt").read_bytes()).hexdigest()

OUT = ROOT/"results/inference"; OUT.mkdir(parents=True, exist_ok=True)
PATCH_DIR = OUT/"field_patches"; PATCH_DIR.mkdir(parents=True, exist_ok=True)
HEAT_DIR = ROOT/"results/heatmaps"; HEAT_DIR.mkdir(parents=True, exist_ok=True)
rows=[]
examples=field_cfg.get("qualitative_examples", [])

for rel in examples:
    rel=Path(rel); group=rel.parts[0]; image_path=ROOT/"pictures"/rel
    if not image_path.exists(): raise FileNotFoundError(image_path)
    patch_file=PATCH_DIR/f"{group}__{image_path.stem}_patches.csv"
    if patch_file.exists():
        patch_df=pd.read_csv(patch_file)
        valid_cache=("model_sha256" in patch_df.columns and len(patch_df) and str(patch_df["model_sha256"].iloc[0])==model_sha256)
    else:
        valid_cache=False
    if valid_cache:
        prob=aggregate_probabilities(patch_df["prob_anomaly"].to_numpy(),field_cfg["aggregation"],field_cfg["topk_fraction"])
        pred=int(prob>=cfg["model"]["threshold"])
        print(f"Reutilizando patches: {image_path.name} ({len(patch_df)})")
    else:
        prob,pred,patch_df=predict_field_image(
            model,image_path,device,input_size=cfg["model"]["input_size"],
            patch_size=field_cfg["patch_size"],overlap=field_cfg["overlap_fraction"],
            threshold=cfg["model"]["threshold"],aggregation=field_cfg["aggregation"],
            topk_fraction=field_cfg["topk_fraction"],batch_size=field_cfg.get("batch_size",64))
        patch_df.insert(0,"file",image_path.name); patch_df.insert(0,"group",group); patch_df["model_sha256"]=model_sha256
        patch_df.to_csv(patch_file,index=False)
    best=patch_df.loc[patch_df.prob_anomaly.idxmax()]
    rows.append({"group":group,"file":image_path.name,"prob_anomaly":prob,"prediction":pred,"n_patches":len(patch_df),
                 "max_patch_prob":float(best.prob_anomaly),"roi_x1":int(best.x1),"roi_y1":int(best.y1),"roi_x2":int(best.x2),"roi_y2":int(best.y2)})
    pd.DataFrame(rows).to_csv(OUT/"field_qualitative_results.csv",index=False)

    heat_file=HEAT_DIR/f"{group}__{image_path.stem}_heatmap.png"
    if not heat_file.exists():
        im=ImageOps.exif_transpose(Image.open(image_path)).convert('RGB'); hmap=probability_map(im.size,patch_df)
        fig,ax=plt.subplots(figsize=(5.5,4)); ax.imshow(im); ax.imshow(hmap,cmap='jet',alpha=float(field_cfg.get('heatmap_alpha',.4)),vmin=0,vmax=1)
        ax.set_title(f"{image_path.name} | P={prob:.3f}"); ax.axis('off'); fig.tight_layout()
        fig.savefig(heat_file,dpi=180,bbox_inches='tight'); plt.close(fig)

results=pd.DataFrame(rows)
results.to_csv(OUT/"field_qualitative_results.csv",index=False)
display(results)

## 4. Resumen de detección

In [ ]:
summary=results[["group","file","prob_anomaly","prediction","n_patches","max_patch_prob"]].copy()
summary.to_csv(ROOT/"results/inference/field_qualitative_summary.csv",index=False)
display(summary.round(4))
print("P6 exportó cuatro ejemplos cualitativos de campo, sus patches y mapas de probabilidad. No se reporta una tasa de detección externa.")